# RoomSplat — Colab GPU fallback

Train a 3D Gaussian Splat from a room **video** on a free Colab GPU — no local install, works on any laptop.

**Before you start:** Runtime → *Change runtime type* → **GPU (T4)**.

Pipeline: upload video → ffmpeg frames → COLMAP poses → train splat → download `scene.ply` → drop into `viewer/index.html`.

> Colab environments drift over time. If a version install fails, the official Nerfstudio Colab is a reliable reference: https://docs.nerf.studio/ — the steps are the same.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install COLMAP + Nerfstudio (splatfacto = Gaussian splatting)

In [ ]:
!apt-get -qq install -y colmap >/dev/null
!pip -q install nerfstudio
# (first run takes several minutes; installs torch + tinycudann + nerfstudio)

## 3. Upload your room video

In [ ]:
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
up = files.upload()           # pick your .mp4
VIDEO = 'data/' + list(up.keys())[0]
print('uploaded', VIDEO)

## 4. Frames + camera poses (Nerfstudio wraps ffmpeg + COLMAP)

In [ ]:
!ns-process-data video --data "$VIDEO" --output-dir proc --num-frames-target 250

## 5. Train the Gaussian splat

In [ ]:
!ns-train splatfacto --data proc --viewer.quit-on-train-completion True --max-num-iterations 30000

## 6. Export `scene.ply` and download it

In [ ]:
import glob
cfg = sorted(glob.glob('outputs/**/config.yml', recursive=True))[-1]
!ns-export gaussian-splat --load-config "$cfg" --output-dir export
from google.colab import files
ply = sorted(glob.glob('export/**/*.ply', recursive=True))[-1]
print('splat:', ply)
files.download(ply)   # -> rename to scene.ply, drop into roomsplat/viewer/